# 04 · Detect everyday objects with pretrained YOLO

**Level:** beginner → intermediate · **Presentation:** 25–40 minutes · **Hardware:** CPU is enough.

**By the end:** run a real pretrained detector, inspect its boxes, explain confidence versus
NMS IoU, and save a result that another program can consume. Prerequisites: notebooks 01–03.
We deliberately use the small, established **YOLO11n** detector, not a claim about the latest
YOLO release. Its COCO vocabulary does not include every object you can imagine.

Run cells from top to bottom. The first normal run downloads official example weights and a
bus photograph; later runs reuse local files. `COURSE_OFFLINE=1` skips model/download sections
explicitly. An offline skip is **not** evidence that real inference passed.

**Presenter opening:** “A detector predicts a class, a location, and a score for each candidate
object. Let us inspect all three before trusting the picture.”


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from course_utils import get_sample_image, load_yolo

OFFLINE = os.getenv("COURSE_OFFLINE", "0") == "1"
SMOKE = os.getenv("COURSE_SMOKE", "0") == "1"
MODEL = "yolo11n.pt"
DEVICE = "cpu"       # Set to 0 only after confirming a CUDA-enabled PyTorch installation.
IMAGE_SIZE = 320     # Try 640 later; small objects may benefit at a latency cost.
CONFIDENCE = 0.25
NMS_IOU = 0.70
OUT = ROOT / "outputs" / "04_pretrained"
OUT.mkdir(parents=True, exist_ok=True)
model = image = result = None
print({"offline": OFFLINE, "model": MODEL, "device": DEVICE, "imgsz": IMAGE_SIZE})


## 1. Inspect the input before predicting

PIL uses RGB; OpenCV generally uses BGR. Here we pass a PIL image to the detector, so the
channel convention is unambiguous. The model will resize and pad the image internally;
reported `xyxy` coordinates are mapped back to the original image dimensions.

**Predict aloud:** name objects you can see, including partially hidden ones. This is a
qualitative inspection, not a labeled evaluation set.


In [ ]:
if OFFLINE:
    print("Skipped: sample download and pretrained model inference (COURSE_OFFLINE=1).")
else:
    image_path = get_sample_image()
    image = Image.open(image_path).convert("RGB")
    print("Cached image:", image_path, "| RGB size (width, height):", image.size)
    display(image.copy().resize((405, round(image.height * 405 / image.width))))


## 2. Run the detector

`conf` filters weak predictions. `iou` controls overlap suppression between competing boxes
in this NMS-based model. They are different thresholds. A confidence of 0.8 is not a guarantee
of 80% correctness on your camera or dataset. Keep thresholds fixed when comparing models.


In [ ]:
if not OFFLINE:
    model = load_yolo(MODEL)
    started = time.perf_counter()
    result = model.predict(image, imgsz=IMAGE_SIZE, conf=CONFIDENCE,
                           iou=NMS_IOU, device=DEVICE, verbose=False)[0]
    print(f"First call: {1000 * (time.perf_counter() - started):.1f} ms (includes initialization)")
    print("Classes available:", len(model.names))
    print("Example class names:", list(model.names.items())[:10])
    print("Detections:", len(result.boxes))


## 3. Turn tensors into a readable table

Each row is a detection. `x1, y1` is the upper-left corner; `x2, y2` is the lower-right.
`.cpu().numpy()` moves results to ordinary arrays so plotting and data analysis work even
if you later choose a GPU. Empty results are valid and must not crash an application.


In [ ]:
def detection_table(prediction):
    columns = ["class_id", "class_name", "confidence", "x1", "y1", "x2", "y2"]
    rows = []
    for box, score, class_id in zip(prediction.boxes.xyxy.cpu().numpy(),
                                     prediction.boxes.conf.cpu().numpy(),
                                     prediction.boxes.cls.cpu().numpy().astype(int)):
        rows.append([int(class_id), prediction.names[int(class_id)], float(score),
                     *[float(v) for v in box]])
    return pd.DataFrame(rows, columns=columns)

detections = pd.DataFrame()
if result is not None:
    detections = detection_table(result)
    display(detections.round(3))
    fig, ax = plt.subplots(figsize=(8, 8))
    show_boxes(np.asarray(image), detections[["x1", "y1", "x2", "y2"]].to_numpy(),
               labels=detections["class_name"].tolist(), scores=detections["confidence"].tolist(),
               ax=ax, title="YOLO11n: inspect boxes, labels, and scores")
    plt.show()


## 4. Change one knob: confidence threshold

First predict what happens at 0.10, 0.25, 0.50, and 0.80. Lower thresholds usually keep more
detections, including uncertain ones. A larger count does not mean better accuracy.
This experiment uses actual model calls; it does not alter the saved baseline result.


In [ ]:
sweep = pd.DataFrame()
if model is not None:
    rows = []
    for threshold in ([0.25, 0.80] if SMOKE else [0.10, 0.25, 0.50, 0.80]):
        prediction = model.predict(image, imgsz=IMAGE_SIZE, conf=threshold,
                                   iou=NMS_IOU, device=DEVICE, verbose=False)[0]
        table = detection_table(prediction)
        rows.append({"confidence_threshold": threshold, "detections": len(table),
                     "class_names": ", ".join(sorted(set(table["class_name"])))})
    sweep = pd.DataFrame(rows)
    display(sweep)
    ax = sweep.plot(x="confidence_threshold", y="detections", marker="o", legend=False,
                    title="Threshold changes retained predictions, not ground truth")
    ax.set_ylabel("Retained detections")
    ax.set_ylim(bottom=0)
    plt.show()


## 5. Filter a class, crop detections, and save an artifact

We look up the class ID by its **name**, avoiding a hidden dependency on a hard-coded index.
Crops are clamped to image bounds. Saved coordinates remain in original-image pixels.
The JSON is suitable for a later app, but predictions still need evaluation and error handling.


In [ ]:
if result is not None:
    person_ids = [class_id for class_id, name in result.names.items() if name == "person"]
    people = detections[detections["class_id"].isin(person_ids)].copy()
    print("Predicted people in this single image:", len(people))
    for number, (_, row) in enumerate(people.head(4).iterrows()):
        x1, y1, x2, y2 = row[["x1", "y1", "x2", "y2"]].to_numpy(dtype=float)
        crop_box = (max(0, int(np.floor(x1))), max(0, int(np.floor(y1))),
                    min(image.width, int(np.ceil(x2))), min(image.height, int(np.ceil(y2))))
        if crop_box[2] > crop_box[0] and crop_box[3] > crop_box[1]:
            crop = image.crop(crop_box)
            crop.thumbnail((160, 240))
            display(crop)
    payload = {"model": MODEL, "image": image_path.name, "width": image.width,
               "height": image.height, "coordinate_format": "xyxy_pixels",
               "confidence_threshold": CONFIDENCE, "nms_iou": NMS_IOU,
               "predictions": detections.to_dict(orient="records")}
    (OUT / "detections.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    detections.to_csv(OUT / "detections.csv", index=False)
    # Ultralytics result.plot() returns BGR; convert before saving through PIL.
    Image.fromarray(result.plot()[..., ::-1]).save(OUT / "annotated.png")
    print("Saved JSON, CSV, and annotated.png to", OUT)
else:
    print("No model artifacts written because inference was skipped.")


## 6. Demonstration challenges

1. Set `IMAGE_SIZE=640`, rerun inference, and inspect small or occluded objects. Do not infer
   accuracy from a single image. Why is the first timing unsuitable as a steady-state benchmark?
2. Compare NMS IoU values 0.30 and 0.90 while keeping confidence fixed. A larger NMS threshold
   permits more overlap; it does **not** require predictions to match ground truth better.
3. Replace the sample with your own image: set `image = Image.open(ROOT / "assets" / "my_photo.jpg").convert("RGB")`
   and `image_path = ROOT / "assets" / "my_photo.jpg"`, then rerun steps 2–5.
4. Explain a miss, a false detection, and a poor box. Which needs a threshold change and which
   might require more representative training data?

**Reveal after discussion — solution to challenge 2:**


In [ ]:
if model is not None:
    comparison = []
    for overlap in [0.30, 0.90]:
        prediction = model.predict(image, imgsz=IMAGE_SIZE, conf=CONFIDENCE,
                                   iou=overlap, device=DEVICE, verbose=False)[0]
        comparison.append({"nms_iou": overlap, "detections": len(prediction.boxes)})
    display(pd.DataFrame(comparison))
    print("Equal counts are possible when boxes do not compete at these IoU values.")


## Presenter checkpoint and troubleshooting

You should be able to point to the original input, a prediction table, an annotated image,
a threshold sweep, and three saved files. The exact boxes can vary with package versions.

| Symptom | Practical action |
|---|---|
| First download fails | Check internet access; rerun after connectivity returns. Do not substitute invented detections. |
| `No module named ultralytics` | Select the course kernel and install the root `requirements.txt`. |
| No detections on your photo | Inspect image orientation, class vocabulary, resolution, then confidence. |
| Wrong colors | Pass a PIL RGB image or respect OpenCV's BGR convention. |
| High first-call latency | Warm up before timing; notebook 12 demonstrates this. |

**Next:** temporal IDs and actual crossing events in notebook 05.

Primary references: [YOLO11](https://docs.ultralytics.com/models/yolo11/),
[prediction arguments and Results](https://docs.ultralytics.com/modes/predict/).
